# CRISP-DM Data Understanding Module
## CDC Diabetes Health Indicators (BRFSS 2015) Dataset

**Author:** Senior Data Analytics Engineer  
**Methodology:** CRISP-DM (Phase 2: Data Understanding)  
**Project:** Diabetes-Analytics  
**Target Variable:** `Diabetes_binary`  

---

### Objective
This notebook performs the initial **Data Understanding** phase for the CDC Diabetes Health Indicators dataset. We inspect and summarize the dataset's structural characteristics, data types, missing values, duplicate records, target class distribution, descriptive statistics, unique values, and range validations without modifying any values.

> **Important Note:** In accordance with clinical epidemiology practices, we analyze the **original imbalanced dataset** to accurately reflect real-world diabetes prevalence. No rebalancing (SMOTE, undersampling, or oversampling) is applied during this stage.

### 1. Setup and Imports
First, we import the necessary scientific libraries and configure path settings using `pathlib` for portability.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Define base directory and paths
BASE_DIR = Path().resolve().parent
DATA_PATH = BASE_DIR / "data" / "raw" / "diabetes_binary_health_indicators_BRFSS2015.csv"
RESULTS_DIR = BASE_DIR / "results" / "data_understanding"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base Directory: {BASE_DIR}")
print(f"Data Path: {DATA_PATH}")
print(f"Results Directory: {RESULTS_DIR}")

### 2. Load Dataset (Task 1)
We load the raw CSV file into a pandas DataFrame.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset successfully loaded. Shape: {df.shape}")
df.head(5)

### 3. Dataset Overview (Task 2)
We summarize the structural dimensions of the dataset and save the summary to a CSV file.

In [ ]:
overview_data = {
    "Metric": [
        "Number of Rows",
        "Number of Columns",
        "Dataset Shape",
        "Column Names"
    ],
    "Value": [
        len(df),
        len(df.columns),
        str(df.shape),
        ", ".join(df.columns.tolist())
    ]
}
overview_df = pd.DataFrame(overview_data)
overview_df.to_csv(RESULTS_DIR / "dataset_overview.csv", index=False)

print("=== Dataset Overview ===")
display(overview_df)

### 4. Data Types Summary (Task 3)
We report the data types of all columns and count the variables by type (Binary, Numerical, Ordinal).

In [ ]:
# Variable classification scheme
COL_CLASSIFICATION = {
    "Diabetes_binary": {"Category": "Target", "Type": "Binary", "Description": "Diabetes status (0 = no diabetes, 1 = prediabetes or diabetes)"},
    "HighBP": {"Category": "Health Condition", "Type": "Binary", "Description": "High blood pressure indicator (0 = no high BP, 1 = high BP)"},
    "HighChol": {"Category": "Health Condition", "Type": "Binary", "Description": "High cholesterol indicator (0 = no high cholesterol, 1 = high cholesterol)"},
    "CholCheck": {"Category": "Healthcare Access", "Type": "Binary", "Description": "Cholesterol check in past 5 years (0 = no check, 1 = check)"},
    "BMI": {"Category": "Physical Measurement", "Type": "Numerical", "Description": "Body Mass Index (BMI)"},
    "Smoker": {"Category": "Lifestyle", "Type": "Binary", "Description": "Smoked at least 100 cigarettes in lifetime (0 = no, 1 = yes)"},
    "Stroke": {"Category": "Health Condition", "Type": "Binary", "Description": "Ever told you had a stroke (0 = no, 1 = yes)"},
    "HeartDiseaseorAttack": {"Category": "Health Condition", "Type": "Binary", "Description": "Coronary heart disease or myocardial infarction (0 = no, 1 = yes)"},
    "PhysActivity": {"Category": "Lifestyle", "Type": "Binary", "Description": "Physical activity in past 30 days excluding work (0 = no, 1 = yes)"},
    "Fruits": {"Category": "Lifestyle", "Type": "Binary", "Description": "Consume fruit 1 or more times per day (0 = no, 1 = yes)"},
    "Veggies": {"Category": "Lifestyle", "Type": "Binary", "Description": "Consume vegetables 1 or more times per day (0 = no, 1 = yes)"},
    "HvyAlcoholConsump": {"Category": "Lifestyle", "Type": "Binary", "Description": "Heavy alcohol consumption (0 = no, 1 = yes)"},
    "AnyHealthcare": {"Category": "Healthcare Access", "Type": "Binary", "Description": "Have any health care coverage (0 = no, 1 = yes)"},
    "NoDocbcCost": {"Category": "Healthcare Access", "Type": "Binary", "Description": "Could not see doctor because of cost in past 12 months (0 = no, 1 = yes)"},
    "GenHlth": {"Category": "General Health", "Type": "Ordinal", "Description": "Self-reported general health scale (1 = excellent to 5 = poor)"},
    "MentHlth": {"Category": "General Health", "Type": "Numerical", "Description": "Days of poor mental health in past 30 days (0-30)"},
    "PhysHlth": {"Category": "General Health", "Type": "Numerical", "Description": "Days of poor physical health in past 30 days (0-30)"},
    "DiffWalk": {"Category": "Health Condition", "Type": "Binary", "Description": "Serious difficulty walking or climbing stairs (0 = no, 1 = yes)"},
    "Sex": {"Category": "Demographic", "Type": "Binary", "Description": "Biological sex (0 = female, 1 = male)"},
    "Age": {"Category": "Demographic", "Type": "Ordinal", "Description": "13-level age category (1 = 18-24 to 13 = 80+)"},
    "Education": {"Category": "Socioeconomic", "Type": "Ordinal", "Description": "Education level scale (1 = never attended school to 6 = college graduate)"},
    "Income": {"Category": "Socioeconomic", "Type": "Ordinal", "Description": "Income scale (1 = <$10,000 to 8 = $75,000+)"}
}

summary_list = []
for col in df.columns:
    info = COL_CLASSIFICATION.get(col, {"Type": "Unknown", "Category": "Unknown"})
    summary_list.append({
        "Column": col,
        "Pandas DataType": str(df[col].dtype),
        "Variable Type": info["Type"],
        "Domain Category": info["Category"]
    })

types_df = pd.DataFrame(summary_list)
print("=== Column Data Types and Categories ===")
display(types_df)

# Print counts
counts = types_df["Variable Type"].value_counts()
print("\n=== Counts of Variables by Type ===")
for var_type, count in counts.items():
    print(f"- {var_type} Variables: {count}")

### 5. Missing Values Analysis (Task 4)
We check for missing counts and percentages per column. If no missing values are found, we explicitly state it.

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": missing_count,
    "Missing Percentage": missing_pct
}).sort_values(by="Missing Percentage", ascending=False)

missing_df.to_csv(RESULTS_DIR / "missing_values.csv", index=False)

print("=== Missing Values Summary ===")
display(missing_df)

total_missing = missing_count.sum()
if total_missing == 0:
    print("\nExplicit Verification: There are absolutely NO missing values (nulls/NaNs) in the dataset.")
else:
    print(f"\nTotal missing values: {total_missing}")

### 6. Duplicate Records Summary (Task 5)
We calculate the total count and percentage of duplicate rows without removing them.

In [ ]:
dup_count = df.duplicated().sum()
dup_pct = (dup_count / len(df)) * 100

dup_df = pd.DataFrame({
    "Metric": ["Total Rows", "Duplicate Rows Count", "Duplicate Percentage"],
    "Value": [len(df), dup_count, f"{dup_pct:.4f}%"]
})
dup_df.to_csv(RESULTS_DIR / "duplicate_summary.csv", index=False)

print("=== Duplicate Records Summary ===")
display(dup_df)
print(f"Interpretation: Out of {len(df):,} respondents, {dup_count:,} duplicate profiles are present ({dup_pct:.4f}%).")
print("These represent identical survey response combinations and are kept to retain the true population distribution.")

### 7. Target Variable Distribution (Task 6)
We analyze the distribution of the target variable `Diabetes_binary` and generate a plot.

In [ ]:
target_counts = df["Diabetes_binary"].value_counts().sort_index()
target_pct = df["Diabetes_binary"].value_counts(normalize=True).sort_index() * 100

dist_df = pd.DataFrame({
    "Class": target_counts.index.astype(int),
    "Count": target_counts.values,
    "Percentage": target_pct.values
})
dist_df.to_csv(RESULTS_DIR / "target_distribution.csv", index=False)

print("=== Target Class Distribution ===")
display(dist_df)

# Plot target distribution with premium styling
plt.figure(figsize=(7, 5))
colors = ["#4A90E2", "#E94E77"]

bars = plt.bar(
    ["0: No Diabetes", "1: Prediabetes/Diabetes"],
    target_counts.values,
    color=colors,
    edgecolor="none",
    width=0.5,
    alpha=0.85
)

# Customize axes
plt.grid(axis="y", linestyle="--", alpha=0.5, color="#CCCCCC")
plt.gca().set_axisbelow(True)
for spine in ["top", "right", "left"]:
    plt.gca().spines[spine].set_visible(False)
plt.gca().spines["bottom"].set_color("#888888")

# Add percentage and value labels
total = len(df)
for bar in bars:
    height = bar.get_height()
    percentage = (height / total) * 100
    plt.text(
        bar.get_x() + bar.get_width()/2.0,
        height + (total * 0.01),
        f"{height:,}\n({percentage:.2f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color="#333333"
    )

plt.title("Distribution of Target Variable: Diabetes_binary", fontsize=13, fontweight="bold", pad=20)
plt.ylabel("Frequency (Count)", fontsize=11)
plt.ylim(0, max(target_counts.values) * 1.15)
plt.tight_layout()

# Save figure
plt.savefig(RESULTS_DIR / "target_distribution.png", dpi=300)
plt.show()

### 8. Summary Statistics (Task 7)
We calculate central tendencies, variance, quartiles, and range for the numerical columns.

In [ ]:
num_cols = [col for col, info in COL_CLASSIFICATION.items() if info["Type"] == "Numerical"]

desc_stats = df[num_cols].describe()
desc_stats.loc["range"] = desc_stats.loc["max"] - desc_stats.loc["min"]
desc_stats = desc_stats.reindex(["count", "mean", "std", "min", "25%", "50%", "75%", "max", "range"])

desc_stats_t = desc_stats.T
desc_stats_t.index.name = "Variable"
desc_stats_t.to_csv(RESULTS_DIR / "descriptive_statistics.csv")

print("=== Descriptive Statistics for Numerical Variables ===")
display(desc_stats_t)

### 9. Unique Values (Task 8)
We report the unique count and values for every attribute in the dataset.

In [ ]:
unique_list = []
for col in df.columns:
    unique_cnt = df[col].nunique()
    sorted_uniques = sorted(df[col].unique())
    
    if unique_cnt <= 15:
        uniques_str = str([int(x) if x.is_integer() else x for x in sorted_uniques])
    else:
        uniques_str = f"[{int(sorted_uniques[0])}, ..., {int(sorted_uniques[-1])}] (Total: {unique_cnt} values)"
        
    unique_list.append({
        "Column": col,
        "Unique Count": unique_cnt,
        "Unique Values": uniques_str
    })

unique_df = pd.DataFrame(unique_list)
unique_df.to_csv(RESULTS_DIR / "unique_values.csv", index=False)

print("=== Unique Values per Column ===")
display(unique_df)

### 10. Range Validation (Task 9)
We validate that all key fields fall strictly within their expected range of CDC values and report any violations.

In [ ]:
validation_rules = {
    "BMI": {"min": 0.0, "max": np.inf, "descr": "Non-negative value"},
    "MentHlth": {"min": 0.0, "max": 30.0, "descr": "0 to 30 days"},
    "PhysHlth": {"min": 0.0, "max": 30.0, "descr": "0 to 30 days"},
    "GenHlth": {"min": 1.0, "max": 5.0, "descr": "1 to 5 scale"},
    "Age": {"min": 1.0, "max": 13.0, "descr": "1 to 13 category scale"},
    "Education": {"min": 1.0, "max": 6.0, "descr": "1 to 6 category scale"},
    "Income": {"min": 1.0, "max": 8.0, "descr": "1 to 8 category scale"}
}

validation_results = []
for col, rules in validation_rules.items():
    min_val = rules["min"]
    max_val = rules["max"]
    actual_min = df[col].min()
    actual_max = df[col].max()
    
    out_of_bounds = df[(df[col] < min_val) | (df[col] > max_val)]
    violation_count = len(out_of_bounds)
    
    status = "PASSED" if violation_count == 0 else "FAILED"
    validation_results.append({
        "Variable": col,
        "Expected Range": rules["descr"],
        "Actual Min": actual_min,
        "Actual Max": actual_max,
        "Violations Count": violation_count,
        "Status": status
    })

validation_df = pd.DataFrame(validation_results)
validation_df.to_csv(RESULTS_DIR / "range_validation.csv", index=False)

print("=== Range Validation Findings ===")
display(validation_df)

failures = validation_df[validation_df["Status"] == "FAILED"]
if len(failures) == 0:
    print("\nAll critical variables successfully PASSED range validation. No unexpected values found.")
else:
    print(f"\nValidation FAILED for {len(failures)} columns. Please investigate outliers.")

### 11. Variable Classification Table (Task 10)
We construct a structured classification mapping of each variable to its type and domain category.

In [ ]:
info_list = []
for var, details in COL_CLASSIFICATION.items():
    info_list.append({
        "Variable": var,
        "Type": details["Type"],
        "Category": details["Category"],
        "Description": details["Description"]
    })
var_info_df = pd.DataFrame(info_list)
var_info_df.to_csv(RESULTS_DIR / "variable_information.csv", index=False)

print("=== Variable Classification Schema ===")
display(var_info_df)

### 12. Verification of Saved Outputs
We print the list of generated files saved under `results/data_understanding/`.

In [ ]:
print("=== Generated Outputs ===")
for file in RESULTS_DIR.iterdir():
    if file.is_file():
        size_kb = file.stat().st_size / 1024
        print(f"- {file.name} ({size_kb:.2f} KB)")